# ÍNDICES e PARTICIONAMENTO

Na aula anterior estudamos os conceitos de **dependências funcionais** e **normalização**. O conhecimento das formas normais propicia verificar a possibilidade de realizar alterações no design das bases de dados, com o objetivo de **evitar repetições** e **recuperar informações de forma fácil**.

Entretanto, mesmo obedecendo (ou não) às formas normais, em certas situações a performance de nossas queries em produção se torna sofrível. Nesta aula, veremos alguns recursos que podemos utilizar para melhorar o desempenho das consultas produzidas.

Este é um tema bastante relevante. Um engenheiro sem o devido conhecimento pode recomendar, por exemplo, a compra de mais servidores (físicos ou na AWS), quando a aplicação dos conceitos vistos nesta aula poderiam gerar ganhos de múltiplas vezes no tempo de execução de queries!

## Instalação da base

Vamos utilizar a base de dados sintética disponível no Blackboard. Execute o script `script_elet_001.sql` para criar a base de dados. Este script apenas faz a **DDL**, os dados gerão gerados de forma aleatória neste notebook.

## Import das bibliotecas

Vamos realizar o import das bibliotecas.

In [25]:
import mysql.connector
import os
import random
import numpy as np
import pandas as pd
from functools import partial
from datetime import datetime, timedelta
from dotenv import load_dotenv
from IPython.display import clear_output

E vamos criar nosso HELPER de conexão com o banco! Perceba que, uma vez configurado o `.env` não precisaremos mais informar usuários, senhas e URLs!

In [26]:
load_dotenv(override=True)

def get_connection_helper():

    def run_db_query(connection, query, args=None, verbose=True):
        with connection.cursor() as cursor:
            if verbose:
                print("Executando query:")
            cursor.execute(query, args)
            for result in cursor:
                if verbose:
                    print(result)

    connection = mysql.connector.connect(
        host=os.getenv("MD_DB_SERVER"),
        port=int(os.getenv("MD_DB_PORT", 3306)),
        user=os.getenv("MD_DB_USERNAME"),
        password=os.getenv("MD_DB_PASSWORD"),
        database="eletrobeer",
    )
    return connection, partial(run_db_query, connection)


connection, db = get_connection_helper()

## Gerar dados para a base

Vamos gerar alguns valores aleatórios e inserir na base de dados `eletrobeer`.

Primeiro, defina a quantidade de linhas a serem inseridas e demais variáveis de ambiente:

In [27]:
QTDE_PED = 6000000
BATCH_SIZE = 200000

START_DATE = datetime(2015, 1, 1)
END_DATE = datetime(2022, 12, 31)

CIDADES_UF = np.array([
    ["São Paulo", "SP"],
    ["Campinas", "SP"],
    ["Ribeirão Preto", "SP"],
    ["São Roque", "SP"],
    ["Rio de Janeiro", "RJ"],
    ["Macaé", "RJ"],
    ["Angra dos Reis", "RJ"],
])
NUM_CIDADES = len(CIDADES_UF)
DATE_DELTA_DAYS = (END_DATE - START_DATE).days

Vamos definir algumas funções auxiliares

In [28]:
def gerar_lote_pedidos(start_id, batch_size):
    """Gera um lote de pedidos com dados aleatórios."""
    ids_pedido = np.arange(start_id, start_id + batch_size, dtype=np.int32)
    ids_cliente = np.random.randint(1000, 10001, size=batch_size, dtype=np.int32)
    qtde_itens = np.random.randint(1, 201, size=batch_size, dtype=np.int32)
    valor_total = np.random.uniform(15.0, 300000.0, size=batch_size)
    
    random_days = np.random.randint(0, DATE_DELTA_DAYS + 1, size=batch_size)
    datas_criacao = (
        pd.to_datetime(START_DATE) + pd.to_timedelta(random_days, unit='D')
    ).strftime("%Y-%m-%d").to_numpy()
    
    indices_cidade = np.random.randint(0, NUM_CIDADES, size=batch_size)
    cidades = CIDADES_UF[indices_cidade, 0]
    ufs = CIDADES_UF[indices_cidade, 1]

    pedidos = list(zip(
        ids_pedido.tolist(),
        ids_cliente.tolist(),
        datas_criacao.tolist(),
        qtde_itens.tolist(),
        valor_total.tolist(),
        cidades.tolist(),
        ufs.tolist()
    ))
    
    return pedidos


def insert_pedidos(connection, pedidos):
    """Insere um lote de pedidos na tabela `pedido`."""
    with connection.cursor() as cursor:
        sql = """INSERT INTO eletrobeer.pedido
        (id_pedido, id_cliente, data_criacao, qtde_itens, valor_total, cidade_entrega, uf_entrega)
        VALUES (%s, %s, %s, %s, %s, %s, %s)"""
        cursor.executemany(sql, pedidos)

Garantir que a tabela `pedido` está vazia

In [29]:
db("TRUNCATE eletrobeer.pedido")

Executando query:


Então, geramos os dados aleatórios.

**<span style="color:red">Atenção</span>**: Este processo pode demorar alguns minutos! A cada aproximadamente 5 segundos, `BATCH_SIZE` inserções devem ser realizadas, com o retorno da mensagem `Completou 800,000 pedidos (Lote 1 de 30)` e assim por diante!

**Dica:** enquanto executa a próxima célula, apenas por curiosidade, leia o seguinte material https://dev.mysql.com/doc/refman/8.0/en/optimizing-innodb-bulk-data-loading.html.

In [30]:
num_batches = (QTDE_PED + BATCH_SIZE - 1) // BATCH_SIZE

print(f"Iniciando a geração e processamento de {QTDE_PED:,} pedidos em {num_batches} lotes de {BATCH_SIZE:,}...")

for i in range(num_batches):
    start_id = i * BATCH_SIZE + 1
    
    current_batch_size = min(BATCH_SIZE, QTDE_PED - (start_id - 1))
    
    pedidos = gerar_lote_pedidos(start_id, current_batch_size)
    
    if connection:
        insert_pedidos(connection, pedidos)
        
    if (i + 1) * BATCH_SIZE % BATCH_SIZE == 0:
        clear_output(wait=True)
        print(f"Completou {(i + 1) * BATCH_SIZE:,} pedidos (Lote {i+1} de {num_batches})")

print(f"Processamento de {QTDE_PED:,} pedidos concluído.")

Completou 6,000,000 pedidos (Lote 30 de 30)
Processamento de 6,000,000 pedidos concluído.


E vamos fazer `commit` para garantir que os dados foram salvos!

In [31]:
connection.commit()

## Consultando a base

Vamos fazer algumas consultas. Como nosso objetivo é avaliar a performance das queries, precisamos analisar o tempo que cada query necessita para executar.

Uma opção é fazer o cálculo direto no Python:

In [32]:
import time

start_time = time.time() # get current time

db("SELECT COUNT(*) FROM pedido")

end_time = time.time() # get current time again

time_spent = end_time - start_time # calculate time spent

print("Time spent:", time_spent, "seconds")

Executando query:
(6000000,)
Time spent: 0.3955709934234619 seconds


Entretanto, desta forma estamos considerando o tempo total, incluindo o tempo gasto pelas funções do próprio Python. Talvez não seja um tempo significativo, mas como poderia ser, melhor evitar e considerar o tempo isolado: apenas o que foi gasto de fato para a query executar (após ter sido recebida pelo RDBMS MySQL).

Para isto, vamos ativar **profiling**:

In [33]:
db("SET profiling = 1;", verbose=False)

**Obs**: quando quiser desativar, utilize

```mysql
SET profiling = 0;
```

Então podemos executar algum **SQL**

In [34]:
db("SELECT COUNT(*) FROM pedido")

Executando query:
(6000000,)


Um outro exemplo de **SQL**

In [35]:
db("SELECT * FROM pedido ORDER BY cidade_entrega ASC LIMIT 1")

Executando query:
(1, 7867, datetime.date(2022, 1, 15), 59, Decimal('51531.01'), 'Angra dos Reis', 'RJ')


E ver, no segundo elemento de cada tupla, qual o tempo gasto para a query ser executada.

In [36]:
db("SHOW PROFILES;")

Executando query:
(1, 0.315751, 'SELECT COUNT(*) FROM pedido')
(2, 2.74010525, 'SELECT * FROM pedido ORDER BY cidade_entrega ASC LIMIT 1')


Então, podemos executar novas queries e ver o seu tempo consumido:

**Dica**: aqui, `verbose=False` executará a query, mas não exibirá o resultado na saída. Podemos fazer assim quando nosso interesse é em apenas ter o tempo da query e não o resultado

In [37]:
sql = "SELECT COUNT(*) as QTDE_LINHAS FROM pedido"

db(sql, verbose=False)
db("SHOW PROFILES;")

Executando query:
(1, 0.315751, 'SELECT COUNT(*) FROM pedido')
(2, 2.74010525, 'SELECT * FROM pedido ORDER BY cidade_entrega ASC LIMIT 1')
(3, 0.22098, 'SELECT COUNT(*) as QTDE_LINHAS FROM pedido')


Utilize `SET PROFILING_HISTORY_SIZE = ???;`, trocando `???` pelo **número de queries** que quer ver no resultado dos profiles, para limitar a exibição as últimas **número de queries**.

In [38]:
db("SET PROFILING_HISTORY_SIZE = 2;", verbose=False)
db("SHOW PROFILES;")

Executando query:
(3, 0.22098, 'SELECT COUNT(*) as QTDE_LINHAS FROM pedido')
(4, 0.0007165, 'SET PROFILING_HISTORY_SIZE = 2')


Vamos deixar configurado para `6`. Você pode alterar quando quiser!

In [39]:
db("SET PROFILING_HISTORY_SIZE = 6;", verbose=False)
db("SHOW PROFILES;")

Executando query:
(3, 0.22098, 'SELECT COUNT(*) as QTDE_LINHAS FROM pedido')
(4, 0.0007165, 'SET PROFILING_HISTORY_SIZE = 2')
(5, 0.000417, 'SET PROFILING_HISTORY_SIZE = 6')


### Explorando novos exemplos

Vamos executar algumas consultas e verificar seus tempos de execução.

**Dica**: leia cada query e tente entender o que ela faz!

In [41]:
sql1 = """SELECT count(*) FROM pedido p
WHERE p.cidade_entrega = "São Paulo";"""

sql2 = """SELECT count(*) FROM pedido p
WHERE p.cidade_entrega = "Rio de Janeiro";"""

sql3 = """SELECT count(*) FROM pedido p
WHERE p.cidade_entrega IN ("Rio de Janeiro", "São Paulo");"""

sql4 = """SELECT count(*) FROM pedido p
WHERE p.cidade_entrega LIKE "São%";"""

sql5 = r"""SELECT count(*) FROM pedido p
WHERE p.cidade_entrega LIKE "%de%";"""

sql6 = "SELECT * FROM pedido ORDER BY cidade_entrega DESC LIMIT 5"

db(sql1)
db(sql2)
db(sql3)
db(sql4)
db(sql5)
db(sql6)

db("SHOW PROFILES;")

Executando query:
(855376,)
Executando query:
(856887,)
Executando query:
(1712263,)
Executando query:
(1713964,)
Executando query:
(856887,)
Executando query:
(25, 9084, datetime.date(2021, 10, 3), 147, Decimal('264608.07'), 'São Roque', 'SP')
(2, 1945, datetime.date(2015, 4, 23), 31, Decimal('32995.34'), 'São Roque', 'SP')
(9, 4366, datetime.date(2016, 6, 29), 155, Decimal('168484.62'), 'São Roque', 'SP')
(48, 3741, datetime.date(2022, 7, 15), 71, Decimal('204129.76'), 'São Roque', 'SP')
(34, 5357, datetime.date(2018, 12, 2), 187, Decimal('247775.17'), 'São Roque', 'SP')
Executando query:
(12, 2.0059705, 'SELECT count(*) FROM pedido p\nWHERE p.cidade_entrega = "São Paulo"')
(13, 1.744802, 'SELECT count(*) FROM pedido p\nWHERE p.cidade_entrega = "Rio de Janeiro"')
(14, 1.75080525, 'SELECT count(*) FROM pedido p\nWHERE p.cidade_entrega IN ("Rio de Janeiro", "São Paulo")')
(15, 1.247447, 'SELECT count(*) FROM pedido p\nWHERE p.cidade_entrega LIKE "São%"')
(16, 1.480173, 'SELECT count(*)

### MEGADADOS?!

Podemos perceber que esta base tem (deveria ter!!!) 4 milhões de linhas.

In [42]:
sql = "SELECT COUNT(*) as QTDE_LINHAS FROM pedido"

db(sql)

Executando query:
(6000000,)


No resultados do `SHOW PROFILES`, vemos que o tempo é quase zero. Mas não se engane, estamos mantendo um ambiente controlado, com apenas uma tabela e realizando queries simples o suficiente para entendermos o que está acontecendo.

Em uma situação do mercado profissional, prepare-se para trabalhar com milhões ou bilhões de linhas em dezenas ou centenas de tabelas, construindo queries de centenas e até milhares de linhas que rodam milhares ou milhões de vezes! As situações de dependência entre múltiplas colunas e tabelas gerará muitas situações onde os conceitos da aula poderão ser aplicados.

**Dica**: No MySQL Workbench, dê botão direito na tabela / Table Inspector. Abra o explorer e confira o tamanho do arquivo contido no **Data path**! Novamente, em uma situação de mercado, espere gigabytes!

### Índices

Índices são estruturas de dados que facilitam a localização de informação no banco de dados.

**Obs**: Link para simular `BTREE` https://www.cs.usfca.edu/~galles/visualization/BTree.html

Para criar um índice, vamos utilizar a sintaxe:
```mysql
CREATE INDEX index_name [index_type] 
 ON tbl_name (index_col_name,...)
```

Por exemplo:

```mysql
-- Por padrão, o index será BTREE
CREATE INDEX pedido_cidade_entrega_IDX
 ON eletrobeer.pedido (cidade_entrega);
```

É importante lembrar que nem todos os engines suportam índice **HASH**. A engine padrão da versão que utilizamos (**InnoDB**) não suporta!

In [43]:
db("SHOW ENGINES;")

Executando query:
('ndbcluster', 'NO', 'Clustered, fault-tolerant tables', None, None, None)
('MEMORY', 'YES', 'Hash based, stored in memory, useful for temporary tables', 'NO', 'NO', 'NO')
('InnoDB', 'DEFAULT', 'Supports transactions, row-level locking, and foreign keys', 'YES', 'YES', 'YES')
('PERFORMANCE_SCHEMA', 'YES', 'Performance Schema', 'NO', 'NO', 'NO')
('MyISAM', 'YES', 'MyISAM storage engine', 'NO', 'NO', 'NO')
('FEDERATED', 'NO', 'Federated MySQL storage engine', None, None, None)
('ndbinfo', 'NO', 'MySQL Cluster system information storage engine', None, None, None)
('MRG_MYISAM', 'YES', 'Collection of identical MyISAM tables', 'NO', 'NO', 'NO')
('BLACKHOLE', 'YES', '/dev/null storage engine (anything you write to it disappears)', 'NO', 'NO', 'NO')
('CSV', 'YES', 'CSV storage engine', 'NO', 'NO', 'NO')
('ARCHIVE', 'YES', 'Archive storage engine', 'NO', 'NO', 'NO')


Vamos criar um índice na coluna `cidade_entrega` e repetir as queries.

Mas antes, vamos consultar se existe algum índice atualmente nesta tabela!

In [44]:
db("SHOW INDEX FROM pedido;")

Executando query:
('pedido', 0, 'PRIMARY', 1, 'id_pedido', 'A', 5767237, None, None, '', 'BTREE', '', '', 'YES', None)


Perceba que já existe um índice. Por que ele está aí se ainda não criamos nenhum?!

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Para PK e UNIQUE o index ja e criado automaticamente

</div>

<a href="#" title="O índice foi criado para a coluna que é chave primária! Isto é feito por padrão, uma vez que a chave acaba sendo utilizada em diversas consultas e joins.">Pare o  mouse aqui para ver a resposta</a>

Então criamos o índice na coluna `cidade_entrega`:

In [45]:
db("CREATE INDEX pedido_cidade_entrega_IDX ON eletrobeer.pedido (cidade_entrega);")

Executando query:


E conferimos novamente os índices existentes

In [46]:
db("SHOW INDEX FROM pedido;")

Executando query:
('pedido', 0, 'PRIMARY', 1, 'id_pedido', 'A', 5767237, None, None, '', 'BTREE', '', '', 'YES', None)
('pedido', 1, 'pedido_cidade_entrega_IDX', 1, 'cidade_entrega', 'A', 6, None, None, 'YES', 'BTREE', '', '', 'YES', None)


Caso queira ver o título das coluas, execute o `SHOW INDEX FROM pedido;` direto no MySQL WorkBench!

Agora repetimos as queries. Compare o tempo necessários para suas execuções.

In [47]:
sql1 = """SELECT count(*) FROM pedido p
WHERE p.cidade_entrega = "São Paulo";"""

sql2 = """SELECT count(*) FROM pedido p
WHERE p.cidade_entrega = "Rio de Janeiro";"""

sql3 = """SELECT count(*) FROM pedido p
WHERE p.cidade_entrega IN ("Rio de Janeiro", "São Paulo");"""

sql4 = """SELECT count(*) FROM pedido p
WHERE p.cidade_entrega LIKE "São%";"""

sql5 = r"""SELECT count(*) FROM pedido p
WHERE p.cidade_entrega LIKE "%de%";"""

sql6 = "SELECT * FROM pedido ORDER BY cidade_entrega DESC LIMIT 5"

db(sql1)
db(sql2)
db(sql3)
db(sql4)
db(sql5)
db(sql6)

db("SHOW PROFILES;")

Executando query:
(855376,)
Executando query:
(856887,)
Executando query:
(1712263,)
Executando query:
(1713964,)
Executando query:
(856887,)
Executando query:
(5999982, 9117, datetime.date(2021, 9, 22), 94, Decimal('119460.36'), 'São Roque', 'SP')
(5999978, 1390, datetime.date(2020, 11, 10), 134, Decimal('182429.24'), 'São Roque', 'SP')
(5999950, 1476, datetime.date(2022, 3, 7), 123, Decimal('236099.08'), 'São Roque', 'SP')
(5999944, 5260, datetime.date(2016, 5, 11), 52, Decimal('257578.41'), 'São Roque', 'SP')
(5999933, 4892, datetime.date(2020, 8, 1), 165, Decimal('45458.98'), 'São Roque', 'SP')
Executando query:
(23, 0.23779725, 'SELECT count(*) FROM pedido p\nWHERE p.cidade_entrega = "São Paulo"')
(24, 0.20975225, 'SELECT count(*) FROM pedido p\nWHERE p.cidade_entrega = "Rio de Janeiro"')
(25, 0.7133425, 'SELECT count(*) FROM pedido p\nWHERE p.cidade_entrega IN ("Rio de Janeiro", "São Paulo")')
(26, 0.34229825, 'SELECT count(*) FROM pedido p\nWHERE p.cidade_entrega LIKE "São%"')
(

Compare o tempo **antes** *versus* **depois** da criação do índice. Apesar de não estamos analisando uma amostra de tamanho 1 (CDados manda oi!), é esperado que obtenha uma melhora de múltiplas vezes em algumas queries, mas em outras nem tanto. Você consegue explicar o por que?!

**Dica**: https://dev.mysql.com/doc/refman/8.0/en/index-btree-hash.html#btree-index-characteristics

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Sim, estamos fazendo uma busca binaria em uma arvore onde cada item guarda um ponteiro para o sua linha na tabela original, certas queries como ORDER DESC ja temos uma arvore ordenada, ou seja basta pegar os 5 ultimos (DESC) itens e retornar, ao contrario sem index o sql precisaria percorrer toda a tabela, ordenar e depois retornar os 5 primeiros.

</div>

#### Testando com outros campos e queries!

Vamos testar com com outros campos e queries!

In [48]:
db("SET PROFILING_HISTORY_SIZE = 4;", verbose=False)

In [49]:
sql1 = """SELECT count(*) FROM pedido p
WHERE p.qtde_itens = 8;"""

sql2 = """SELECT count(*) FROM pedido p
WHERE p.qtde_itens < 5;"""

sql3 = """SELECT count(*) FROM pedido p
WHERE p.qtde_itens BETWEEN 5 AND 20;"""

sql4 = """SELECT count(*) FROM pedido p
WHERE p.qtde_itens IN (8, 15, 20, 45);"""

db(sql1)
db(sql2)
db(sql3)
db(sql4)

db("SHOW PROFILES;")

Executando query:
(29905,)
Executando query:
(119586,)
Executando query:
(480196,)
Executando query:
(119788,)
Executando query:
(30, 1.743263, 'SELECT count(*) FROM pedido p\nWHERE p.qtde_itens = 8')
(31, 1.2874645, 'SELECT count(*) FROM pedido p\nWHERE p.qtde_itens < 5')
(32, 1.327371, 'SELECT count(*) FROM pedido p\nWHERE p.qtde_itens BETWEEN 5 AND 20')
(33, 1.424986, 'SELECT count(*) FROM pedido p\nWHERE p.qtde_itens IN (8, 15, 20, 45)')


Conferindo os índices atuais

In [50]:
db("SHOW INDEX FROM pedido;")

Executando query:
('pedido', 0, 'PRIMARY', 1, 'id_pedido', 'A', 5767237, None, None, '', 'BTREE', '', '', 'YES', None)
('pedido', 1, 'pedido_cidade_entrega_IDX', 1, 'cidade_entrega', 'A', 6, None, None, 'YES', 'BTREE', '', '', 'YES', None)


Crie um index **BTREE** baseado na coluna `qtde_itens`

In [51]:
db("CREATE INDEX idxQtdeItens ON eletrobeer.pedido (qtde_itens)")

Executando query:


Conferindo os índices

In [52]:
db("SHOW INDEX FROM pedido;")

Executando query:
('pedido', 0, 'PRIMARY', 1, 'id_pedido', 'A', 5767237, None, None, '', 'BTREE', '', '', 'YES', None)
('pedido', 1, 'pedido_cidade_entrega_IDX', 1, 'cidade_entrega', 'A', 6, None, None, 'YES', 'BTREE', '', '', 'YES', None)
('pedido', 1, 'idxQtdeItens', 1, 'qtde_itens', 'A', 201, None, None, 'YES', 'BTREE', '', '', 'YES', None)


Repetindo as consultas

In [53]:
sql1 = """SELECT count(*) FROM pedido p
WHERE p.qtde_itens = 8;"""

sql2 = """SELECT count(*) FROM pedido p
WHERE p.qtde_itens < 5;"""

sql3 = """SELECT count(*) FROM pedido p
WHERE p.qtde_itens BETWEEN 5 AND 20;"""

sql4 = """SELECT count(*) FROM pedido p
WHERE p.qtde_itens IN (8, 15, 20, 45);"""

db(sql1)
db(sql2)
db(sql3)
db(sql4)

db("SHOW PROFILES;")

Executando query:
(29905,)
Executando query:
(119586,)
Executando query:
(480196,)
Executando query:
(119788,)
Executando query:
(37, 0.00714925, 'SELECT count(*) FROM pedido p\nWHERE p.qtde_itens = 8')
(38, 0.02202925, 'SELECT count(*) FROM pedido p\nWHERE p.qtde_itens < 5')
(39, 0.09869375, 'SELECT count(*) FROM pedido p\nWHERE p.qtde_itens BETWEEN 5 AND 20')
(40, 0.019937, 'SELECT count(*) FROM pedido p\nWHERE p.qtde_itens IN (8, 15, 20, 45)')


Compare o tempo **antes** *versus* **depois** da criação do índice. Emocionante, não?!

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Emocionante!!!

</div>

## Particionamento

Particionar é dividir as tabelas de um banco de dados em partes menores.

Permite distribuir o banco de dados em vários nós ou HDs diferentes, aumentando o desempenho em situações de acesso concorrente intenso (que não é o nosso caso).

Leia mais em https://dev.mysql.com/doc/refman/8.0/en/partitioning-pruning.html

Veja um exemplo de particionamento:

In [54]:
sql1 = """
SELECT 
    YEAR(p.data_criacao), AVG(valor_total) AS media
FROM
    pedido p
WHERE YEAR(p.data_criacao) > 2020
GROUP BY YEAR(p.data_criacao)
ORDER BY YEAR(p.data_criacao) ASC;"""

db(sql1)

db("SHOW PROFILES;")

Executando query:
(2021, Decimal('149988.386184'))
(2022, Decimal('149842.319423'))
Executando query:
(38, 0.02202925, 'SELECT count(*) FROM pedido p\nWHERE p.qtde_itens < 5')
(39, 0.09869375, 'SELECT count(*) FROM pedido p\nWHERE p.qtde_itens BETWEEN 5 AND 20')
(40, 0.019937, 'SELECT count(*) FROM pedido p\nWHERE p.qtde_itens IN (8, 15, 20, 45)')
(41, 2.1234165, 'SELECT \n    YEAR(p.data_criacao), AVG(valor_total) AS media\nFROM\n    pedido p\nWHERE YEAR(p.data_criacao) > 2020\nGROUP BY YEAR(p.data_criacao)\nORDER BY YEAR(p.data_criacao) ASC')


Para separar por ano, precisaremos fazer com que a `data_criacao` seja parte da chave primária.

In [55]:
sql = """
ALTER TABLE pedido
DROP PRIMARY KEY;"""

db(sql)

Executando query:


In [56]:
sql = """
ALTER TABLE pedido
ADD PRIMARY KEY (id_pedido, data_criacao);"""

db(sql)

Executando query:


Então, separamos a tabela `pedido` em quatro partições

In [57]:
sql = """
ALTER TABLE pedido
PARTITION BY RANGE(YEAR(data_criacao))
(
    PARTITION p0 VALUES LESS THAN (2016),
    PARTITION p1 VALUES LESS THAN (2018),
    PARTITION p2 VALUES LESS THAN (2020),
    PARTITION p3 VALUES LESS THAN MAXVALUE
);"""

db(sql)

Executando query:


Executando a query novamente

In [58]:
sql1 = """
SELECT 
    YEAR(p.data_criacao), AVG(valor_total) AS media
FROM
    pedido p
WHERE YEAR(p.data_criacao) > 2020
GROUP BY YEAR(p.data_criacao)
ORDER BY YEAR(p.data_criacao) ASC;"""

db(sql1)

db("SHOW PROFILES;")

Executando query:
(2021, Decimal('149988.386184'))
(2022, Decimal('149842.319423'))
Executando query:
(42, 37.1280285, 'ALTER TABLE pedido\nDROP PRIMARY KEY')
(43, 54.42767425, 'ALTER TABLE pedido\nADD PRIMARY KEY (id_pedido, data_criacao)')
(44, 52.13450525, 'ALTER TABLE pedido\nPARTITION BY RANGE(YEAR(data_criacao))\n(\n    PARTITION p0 VALUES LESS THAN (2016),\n    PARTITION p1 VALUES LESS THAN (2018),\n    PARTITION p2 VALUES LESS THAN (2020),\n    PARTITION p3 VALUES LESS THAN MAXVALUE\n)')
(45, 3.1428735, 'SELECT \n    YEAR(p.data_criacao), AVG(valor_total) AS media\nFROM\n    pedido p\nWHERE YEAR(p.data_criacao) > 2020\nGROUP BY YEAR(p.data_criacao)\nORDER BY YEAR(p.data_criacao) ASC')


Anote abaixo suas considerações sobre particionar tabelas!

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Nesse caso aqui pelo q eu entendi agora o sql so vai buscar na faixa p3, mas nesse caso nao consigo tirar conclusoes ainda pois foi basicamente igual o tempo.

</div>

## Exercícios

**Exercício 1**: Explique por que uma hash table é:

**a)** Boa para buscas por valor exato?

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Pois a complexidade é O(1), ou seja podemos acessando uma chave e achar o valor que buscamos.

</div>

**b)** Ruim para buscas por faixas de valor?

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Pois nada esta ordenado, neste caso precisamos percorrer toda hash para achar por exemplos todos valores que sao < 5 (exemplo aleatorio, em um mundo q os valores do nosso hash sao inteiros.).

</div>

<a href="#gab_ex1">Click para ver a resposta</a>

**Exercício 2**: Por que os índices de bancos de dados relacionais utilizam `B-tree` e suas variantes ao invés de uma árvore binária balanceada?

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Aparentemente pela profundidade da arvore pois em uma arvoria binaria balanceada cada no so armazena uma chave, enquanto em uma B-tree um nó armazena varias chaves e ponteiros, deixando a profundidade da arvore pequena mesmo armazenando muitos dados.

</div>

**Exercício 3**: O professor demonstrou a construção de uma `B-tree`. Pesquise o que são as `B+-trees` e responda:

**Obs:**
- O `+` representa um **plus**. Já o `-` é só um traço separador!
- Você pode simular a versão **plus** aqui https://www.cs.usfca.edu/~galles/visualization/BPlusTree.html

**a)** Qual a diferença entre a `B-tree` e a versão plus?!

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

A diferença é que na B-tree os valores (dados) ficam distribuídos em todos os nós,
enquanto na B+-tree os nós internos contêm apenas as chaves de indexação e
os dados ficam exclusivamente nos nós folha, que ainda são encadeados entre si.
Isso torna a B+-tree mais eficiente para buscas sequenciais e intervalares, motivo pelo qual é usada em sistemas de bancos de dados

</div>

**b)** O MySQL utiliza `B-tree` ou `B+-trees`?

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

B+-trees

</div>

**Exercício 4**: Explique por que uma B-tree é:

**a)** Razoável para buscas por valor exato?

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Uma B-tree é razoável para buscas por valor exato porque suas chaves estão ordenadas e a árvore é balanceada, permitindo percorrer apenas um caminho da raiz até a folha correspondente. Assim, cada passo reduz drasticamente o espaço de busca, resultando em tempo O(log n).

</div>

**b)** Boa para buscas por faixa de valor?

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

A B-tree é boa para buscas por faixa de valor porque suas chaves estão armazenadas em ordem. Após localizar o ponto inicial da faixa, é possível percorrer os nós seguintes em sequência, visitando apenas as chaves que pertencem ao intervalo desejado, o que torna a operação muito mais eficiente que em estruturas não ordenadas.

</div>

**Exercício 5**: Pense em situações onde o **particionamento vertical** é benéfico, e onde é problemático.

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Voltando ao exemplo da aula, é um benéficio particionar verticalmente (por colunas) quando por exemplo desejamos saber a média do preço de venda geral, ou seja precisariamos apenas de uma coluna e aplicar AVG nela para saber esse valor exato. Agora imagine que gostariamos de agrupar essa media por ano, precisariamos dar join novamente com a tabela para conseguir agrupar pelo ano.

</div>

**Exercício 6**: Pense em situações onde o **particionamento horizontal** é benéfico, e onde é problemático.

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

O particionamento horizontal é benéfico quando as consultas acessam apenas subconjuntos de linhas, como dados de um determinado ano ou região.
Ele se torna problemático quando as consultas precisam varrer todas as partições ou quando os dados ficam mal distribuídos, tornando o ganho de desempenho insignificante.

</div>

## Conexão

Vamos fechar a conexão e finalizamos por hoje!

In [34]:
connection.close()

## Referências
- OLIVEIRA, C. H. P, SQL: Curso Prático, Novatec, 2002 CAP 4
- SILBERSCHATZ, A.; KORTH, H. F.; SUDARSHAN, S. DATABASE SYSTEM CONCEPTS, SEVENTH EDITION CAP 4.6
- https://jasimabasheer.com/posts/btrees

## Gabarito

**<div id="gab_ex1">Exercício 1</div>**


<style>
.box {
  padding: 10px;
  border-radius: 5px;
  border-left: 4px solid var(--accent);
  border: 1px solid var(--border);
  background-color: var(--bg);
}

/* Tema Claro */
@media (prefers-color-scheme: light) {
  .box-error {
    --accent: #ef4444; /* Red-500 */
    --border: rgba(239, 68, 68, 0.4);
    --bg: rgba(239, 68, 68, 0.15);
  }

  .box-warn {
    --accent: #f59e0b; /* Amber-500 */
    --border: rgba(245, 158, 11, 0.4);
    --bg: rgba(245, 158, 11, 0.15);
  }
}

/* Tema Escuro */
@media (prefers-color-scheme: dark) {
  .box-error {
    --accent: #f87171; /* Red-400 */
    --border: rgba(248, 113, 113, 0.4);
    --bg: rgba(248, 113, 113, 0.2);
  }

  .box-warn {
    --accent: #fbbf24; /* Amber-400 (mais claro no dark) */
    --border: rgba(251, 191, 36, 0.4);
    --bg: rgba(251, 191, 36, 0.2);
  }
}
</style>

<div class="box box-error">

**a)** Complexide de busca O(1) 😍

**b)** Não mantém relação de ordem 😭
    
</div>